# v9: Geometric Attention — Per-Subbundle Sparse QK Routing
## The Fourth Force: Selective Cross-Position Access Inside the Langevin Loop

**Author:** David Ledbetter  
**Created:** 03/14/2026  
**Lineage:** v8.2 (per-subbundle memory, gated residual, deep supervision) + geometric attention

### Version History

- **v9.0** (03/14/2026): Per-subbundle sparse attention as 4th additive force.

---

### What v9 Adds

v8.2 plateaued at Val BPC 2.65, Val Acc 45% on Shakespeare. The bottleneck:  
the causal convolution applies the **same temporal kernel regardless of content**.  
Token 10 can't selectively query token 3 while ignoring token 7.

v5 tried attention but put it on the **wrong thing** (per-fiber gauge rotation).  
The cross-position mixing stayed content-independent.

v9 puts attention on the **right thing**: cross-position routing, per-subbundle,  
inside the Langevin loop. Each subbundle independently decides which past  
tokens are most relevant via sparse top-k QK similarity.

### The Four Forces

The score function now has four additive forces:
1. **Hopfield gradient** — settle toward nearest memory attractor (per-subbundle)
2. **Causal context force** — smooth local mixing via causal convolution
3. **Lateral inhibition** — competitive dynamics between fiber dims
4. **Selective attention force** — sparse per-subbundle QK retrieval from specific past tokens **(NEW)**

Forces 2 and 4 are complementary:  
- Causal conv = smooth, position-weighted, content-independent baseline  
- Sparse attention = sharp, content-addressed, selective retrieval

Both operate inside the Langevin loop, refining at every settling step.

### Why Per-Subbundle

- The "syntax" subbundle attends to nearby punctuation (local, k=4)  
- The "semantics" subbundle attends to the subject noun 20 tokens back (global, k=8)  
- Small QK matrices (32x32 per subbundle) — far less overfitting risk than v5's full 256x256  
- Respects the orthogonal decomposition

### Relationship to Anthropic's Finding

The Anthropic paper proved attention heads twist manifolds to align coordinate frames.  
Here, each subbundle's QK interaction detects geometric alignment between the current  
token and each past token — which past positions have features that "twist" into  
alignment with the query. The top-k sparsification selects the most aligned positions.  
This is attention as manifold twist detection, not as probability distribution.

In [19]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from typing import Optional, Tuple
from dataclasses import dataclass
import math
import os
import urllib.request
from tqdm.notebook import tqdm

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cpu


---
## 1. Data (Tiny Shakespeare, character-level)

In [20]:
data_dir = os.path.join(os.path.dirname(os.path.abspath("__file__")), "..", "v8_real_text", "data")
data_path = os.path.join(data_dir, "input.txt")

if not os.path.exists(data_path):
    os.makedirs(data_dir, exist_ok=True)
    url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    print("Downloading Tiny Shakespeare...")
    urllib.request.urlretrieve(url, data_path)

with open(data_path, "r") as f:
    text = f.read()

def encode(s): return [ord(c) for c in s]
def decode(t): return "".join(chr(x) for x in t if 0 <= x < 256)

data = torch.tensor(encode(text), dtype=torch.long)
split = int(0.9 * len(data))
train_data, val_data = data[:split], data[split:]
print(f"Train: {len(train_data):,} | Val: {len(val_data):,} chars")

Train: 1,003,854 | Val: 111,540 chars


---
## 2. Configuration

In [21]:
@dataclass
class ArchitectureConfig:
    # === CONFIG HISTORY ===
    # v8.2: Per-subbundle memory, manifold_dim=128, langevin_steps=8, context_expand=2.
    #   Result: Val BPC 2.65, Val Acc 45%, Val PPL 6.3. Plateaued — content-independent mixing.
    # v9.0 (03/14/2026): Add per-subbundle sparse attention as 4th force.
    #   attn_top_k=8: each subbundle attends to 8 most relevant past positions.
    #   langevin_steps 8->6: attention adds compute per step, reduce steps to compensate.
    #   Removed numerical sparsity enforcement (soft-thresholding). Attention handles
    #   routing; Hopfield settling is functionally sparse in attractor space even
    #   though output vectors are dense. Thresholding was discarding useful info.

    fiber_dim: int = 256
    n_subbundles: int = 8
    manifold_dim: int = 128

    vocab_size: int = 256
    max_seq_len: int = 128

    atoms_per_subbundle: int = 128
    k_wta_per_subbundle: int = 16

    n_blocks: int = 4
    context_expand: int = 2
    attn_top_k: int = 8             # v9: sparse attention — attend to top-k past positions per subbundle

    langevin_steps: int = 6          # v9: 8 -> 6 (attention adds compute per step)
    langevin_lr: float = 0.1
    beta_init: float = 1.0
    beta_final: float = 10.0

    sparsity_lambda: float = 0.3
    inhibition_gamma: float = 0.1

    learning_rate: float = 3e-4
    dropout: float = 0.1
    batch_size: int = 16
    seq_len: int = 64
    max_steps: int = 10000          # v9: doubled — model is not overfitting, let it train
    eval_interval: int = 200
    eval_steps: int = 20

    @property
    def subbundle_dim(self):
        assert self.fiber_dim % self.n_subbundles == 0
        return self.fiber_dim // self.n_subbundles


config = ArchitectureConfig()
print(f"Fiber: {config.fiber_dim} = {config.n_subbundles} x {config.subbundle_dim}")
print(f"Attention: top-{config.attn_top_k} per subbundle per Langevin step")
print(f"Blocks: {config.n_blocks}, Langevin: {config.langevin_steps} steps")

Fiber: 256 = 8 x 32
Attention: top-8 per subbundle per Langevin step
Blocks: 4, Langevin: 6 steps


In [22]:
def get_batch(split_data, cfg):
    max_start = len(split_data) - cfg.seq_len - 1
    starts = torch.randint(0, max_start, (cfg.batch_size,))
    return torch.stack([split_data[s:s+cfg.seq_len] for s in starts]).to(device)

---
## 3. Foundation (from v8.2)

In [23]:
class SparseTokenEmbedding(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.embedding = nn.Embedding(cfg.vocab_size, cfg.fiber_dim)
        self.topk_per_sub = max(1, cfg.subbundle_dim // 4)
        self.manifold_coords = nn.Embedding(cfg.max_seq_len, cfg.manifold_dim)

    def forward(self, token_ids):
        B, T = token_ids.shape
        x = self.embedding(token_ids)
        chunks = x.chunk(self.cfg.n_subbundles, dim=-1)
        sparse_chunks = []
        for c in chunks:
            _, idx = torch.topk(c.abs(), self.topk_per_sub, dim=-1)
            mask = torch.zeros_like(c)
            mask.scatter_(-1, idx, 1.0)
            sparse_chunks.append(c * mask)
        x_sparse = torch.cat(sparse_chunks, dim=-1)
        pos = torch.arange(T, device=token_ids.device)
        q = self.manifold_coords(pos).unsqueeze(0).expand(B, -1, -1)
        return x_sparse, q


class PerSubbundleMemoryBank(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        sd, K, A = cfg.subbundle_dim, cfg.n_subbundles, cfg.atoms_per_subbundle
        self.dictionaries = nn.ParameterList(
            [nn.Parameter(torch.randn(A, sd) * 0.02) for _ in range(K)]
        )
        self.routers = nn.ModuleList([
            nn.Sequential(nn.Linear(cfg.manifold_dim + sd, A), nn.SiLU(), nn.Linear(A, A))
            for _ in range(K)
        ])

    def forward(self, q, x):
        cfg = self.cfg
        sd = cfg.subbundle_dim
        x_chunks = x.split(sd, dim=-1)
        M_list = []
        for chunk, dic, router in zip(x_chunks, self.dictionaries, self.routers):
            D_n = F.normalize(dic, dim=-1)
            logits = router(torch.cat([q, chunk], dim=-1))
            _, idx = torch.topk(logits, cfg.k_wta_per_subbundle, dim=-1)
            M_list.append(D_n[idx])
        return M_list

---
## 4. Per-Subbundle Sparse Attention (v9 Core)

Each subbundle independently computes causal attention over past positions:
1. Project subbundle content into Q, K, V (32 -> 32, small projections)
2. Causal QK scores: each position scores against all past positions
3. Sparse top-k: only attend to the k most aligned positions
4. Softmax over selected positions, weighted V sum

The output is the "selective context" — what specific past tokens  
are most geometrically aligned with the current token in each channel.  
This provides the sharp, content-addressed retrieval that the smooth  
causal convolution cannot.

Complexity: O(T * k * K * d_sub) per Langevin step.  
With T=64, k=8, K=8, d_sub=32: ~131K ops vs full attention's ~1M.

In [24]:
class PerSubbundleAttention(nn.Module):
    """Sparse causal attention per subbundle channel.

    Each of the K subbundles has its own Q, K, V projections.
    Attention is causal (each position only sees past) and sparse
    (top-k most aligned positions per query). This is the geometric
    twist detector from the Anthropic paper, decomposed per-subbundle.
    """

    def __init__(self, cfg: ArchitectureConfig):
        super().__init__()
        self.cfg = cfg
        sd = cfg.subbundle_dim
        K = cfg.n_subbundles

        self.W_Q = nn.ModuleList([nn.Linear(sd, sd, bias=False) for _ in range(K)])
        self.W_K = nn.ModuleList([nn.Linear(sd, sd, bias=False) for _ in range(K)])
        self.W_V = nn.ModuleList([nn.Linear(sd, sd, bias=False) for _ in range(K)])

        self.scale = math.sqrt(sd)

    def forward(self, x_seq: torch.Tensor) -> torch.Tensor:
        """Compute sparse causal attention per subbundle.

        Args:
            x_seq: (B, T, D) full sequence state

        Returns:
            attn_out: (B, T, D) selective context from attended positions
        """
        B, T, D = x_seq.shape
        sd = self.cfg.subbundle_dim
        k = self.cfg.attn_top_k

        x_chunks = x_seq.split(sd, dim=-1)
        out_chunks = []

        # Causal mask: position i can only attend to positions 0..i
        causal_mask = torch.triu(
            torch.ones(T, T, device=x_seq.device) * float('-inf'), diagonal=1
        )  # (T, T) — upper triangle is -inf

        for sub_idx, x_sub in enumerate(x_chunks):
            # x_sub: (B, T, sd)
            Q = self.W_Q[sub_idx](x_sub)  # (B, T, sd)
            K = self.W_K[sub_idx](x_sub)  # (B, T, sd)
            V = self.W_V[sub_idx](x_sub)  # (B, T, sd)

            # Causal attention scores
            scores = torch.bmm(Q, K.transpose(-2, -1)) / self.scale  # (B, T, T)
            scores = scores + causal_mask.unsqueeze(0)  # apply causal mask

            # Sparse top-k: only keep k highest scores per query position
            if k < T:
                topk_vals, topk_idx = torch.topk(scores, k, dim=-1)  # (B, T, k)
                sparse_scores = torch.full_like(scores, float('-inf'))
                sparse_scores.scatter_(-1, topk_idx, topk_vals)
            else:
                sparse_scores = scores

            # Softmax over selected positions (stable — -inf positions get 0 weight)
            attn_weights = F.softmax(sparse_scores, dim=-1)  # (B, T, T)

            # Weighted sum of values
            attn_out_sub = torch.bmm(attn_weights, V)  # (B, T, sd)
            out_chunks.append(attn_out_sub)

        return torch.cat(out_chunks, dim=-1)  # (B, T, D)

---
## 5. Score Function with Four Forces

In [25]:
class CausalScoreFunction(nn.Module):
    """Four additive forces: Hopfield + causal conv + inhibition + sparse attention (v9)."""

    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg

        # Force 2: Causal convolution
        self.log_decay = nn.Parameter(torch.linspace(-1.0, 2.0, cfg.fiber_dim))
        expanded = cfg.fiber_dim * cfg.context_expand
        self.context_proj = nn.Sequential(
            nn.Linear(cfg.fiber_dim, expanded), nn.SiLU(),
            nn.Dropout(cfg.dropout), nn.Linear(expanded, cfg.fiber_dim),
        )

        # Force 3: Lateral inhibition
        self.W_inh = nn.Parameter(torch.randn(cfg.fiber_dim, cfg.fiber_dim) * 0.01)

        # Force 4: Per-subbundle sparse attention (v9)
        self.subbundle_attn = PerSubbundleAttention(cfg)
        self.attn_proj = nn.Sequential(
            nn.Linear(cfg.fiber_dim, cfg.fiber_dim), nn.Tanh(),
        )

        # Per-step scheduling for forces 2 and 4
        # Context (causal conv): high early, low late
        self.step_context_logits = nn.Parameter(
            torch.linspace(1.0, -1.0, cfg.langevin_steps)
        )
        # Attention: also high early, low late (explore then commit)
        self.step_attn_logits = nn.Parameter(
            torch.linspace(0.5, -0.5, cfg.langevin_steps)
        )

    def _causal_mix(self, x_seq):
        B, T, D = x_seq.shape
        decay = F.softplus(self.log_decay)
        lags = torch.arange(T, device=x_seq.device).float()
        kernel = torch.exp(-decay.unsqueeze(0) * lags.unsqueeze(-1))
        kernel = kernel / (kernel.sum(0, keepdim=True) + 1e-8)
        x_p = F.pad(x_seq, (0, 0, 0, T))
        k_p = F.pad(kernel, (0, 0, 0, T))
        X = torch.fft.fft(x_p, dim=1)
        K = torch.fft.fft(k_p, dim=0).unsqueeze(0)
        return torch.fft.ifft(X * K, dim=1).real[:, :T, :]

    def hopfield_gradient_subbundle(self, x_chunk, M_k, beta):
        sim = torch.bmm(M_k, x_chunk.unsqueeze(-1)).squeeze(-1)
        w = F.softmax(beta * sim, dim=-1)
        return -torch.bmm(w.unsqueeze(1), M_k).squeeze(1)

    def forward(self, x_seq, M_q_all_list, beta, step_idx):
        B, T, D = x_seq.shape
        sd = self.cfg.subbundle_dim

        # Force 1: Per-subbundle Hopfield gradient (settle)
        x_flat = x_seq.reshape(B * T, D)
        x_chunks = x_flat.split(sd, dim=-1)
        grad_chunks = [
            self.hopfield_gradient_subbundle(xc, mk, beta)
            for xc, mk in zip(x_chunks, M_q_all_list)
        ]
        grad_E = torch.cat(grad_chunks, dim=-1).reshape(B, T, D)

        # Force 2: Causal context (smooth local mixing)
        x_mixed = self._causal_mix(x_seq)
        context_diff = x_mixed - x_seq
        causal_force = self.context_proj(context_diff.reshape(-1, D)).reshape(B, T, D)
        alpha_ctx = torch.sigmoid(self.step_context_logits[step_idx])
        causal_force = alpha_ctx * causal_force

        # Force 3: Lateral inhibition
        inhibition = self.cfg.inhibition_gamma * (x_flat @ self.W_inh.T).reshape(B, T, D)

        # Force 4: Per-subbundle sparse attention (selective retrieval) [v9]
        attn_context = self.subbundle_attn(x_seq)        # (B, T, D)
        attn_diff = attn_context - x_seq                 # what attention adds
        attn_force = self.attn_proj(attn_diff.reshape(-1, D)).reshape(B, T, D)
        alpha_attn = torch.sigmoid(self.step_attn_logits[step_idx])
        attn_force = alpha_attn * attn_force

        return grad_E + causal_force + inhibition + attn_force

---
## 6. Langevin, Block, CLM (from v8.2)

In [26]:
class SequenceLangevinDescent(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.score_fn = CausalScoreFunction(cfg)

    def forward(self, x_seq, M_q_all, return_trajectory=False):
        cfg = self.cfg
        x = x_seq.clone()
        trajectory = [x.detach().clone()] if return_trajectory else None
        betas = torch.linspace(cfg.beta_init, cfg.beta_final, cfg.langevin_steps)
        ramp = torch.clamp(torch.linspace(-1.0, 1.0, cfg.langevin_steps), min=0.0)

        # v9: Removed numerical sparsity enforcement. Attention handles routing;
        # the Hopfield settling is functionally sparse in attractor space
        # (few atoms dominate) even though the output vector is dense.
        # Soft-thresholding was discarding useful information.
        for step in range(cfg.langevin_steps):
            beta_t = betas[step].item()
            score = self.score_fn(x, M_q_all, beta_t, step)
            noise = math.sqrt(2.0 * cfg.langevin_lr / beta_t) * torch.randn_like(x)
            x = x - cfg.langevin_lr * score + noise
            if return_trajectory:
                trajectory.append(x.detach().clone())
        if return_trajectory:
            trajectory.append(x.detach().clone())
            return x, trajectory
        return x


class DiffusionRoutingBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.memory_bank = PerSubbundleMemoryBank(cfg)
        self.seq_langevin = SequenceLangevinDescent(cfg)
        self.norm = nn.LayerNorm(cfg.fiber_dim)
        self.dropout = nn.Dropout(cfg.dropout)
        self.residual_gate = nn.Sequential(
            nn.Linear(cfg.fiber_dim * 2, cfg.fiber_dim), nn.Sigmoid()
        )

    def forward(self, x_seq, q_coords, return_trajectory=False):
        B, T, D = x_seq.shape
        q_flat = q_coords.reshape(B * T, -1)
        x_flat = x_seq.reshape(B * T, D)
        M_q_all = self.memory_bank(q_flat, x_flat)

        if return_trajectory:
            x_settled, traj = self.seq_langevin(x_seq, M_q_all, return_trajectory=True)
        else:
            x_settled = self.seq_langevin(x_seq, M_q_all)
            traj = None

        gate_in = torch.cat([x_settled, x_seq], dim=-1)
        gate = self.residual_gate(gate_in.reshape(-1, D * 2)).reshape(B, T, D)
        x_out = self.norm(gate * self.dropout(x_settled) + (1 - gate) * x_seq)

        if return_trajectory:
            return x_out, traj
        return x_out,


class SpectralGaugeCLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.embedding = SparseTokenEmbedding(cfg)
        self.blocks = nn.ModuleList([DiffusionRoutingBlock(cfg) for _ in range(cfg.n_blocks)])
        self.decoder = nn.Sequential(
            nn.Linear(cfg.fiber_dim, cfg.fiber_dim), nn.SiLU(),
            nn.Dropout(cfg.dropout), nn.Linear(cfg.fiber_dim, cfg.vocab_size),
        )
        self.register_buffer("block_loss_weights", torch.linspace(0.1, 1.0, cfg.n_blocks))

    def forward(self, token_ids):
        B, T = token_ids.shape
        x, q = self.embedding(token_ids)
        intermediate_logits = []
        for block in self.blocks:
            result = block(x, q)
            x = result[0]
            intermediate_logits.append(self.decoder(x)[:, :-1, :])
        info = {
            "mean_sparsity": (x.abs() < 1e-3).float().mean().item(),
            "intermediate_logits": intermediate_logits,
            "block_loss_weights": self.block_loss_weights,
        }
        return intermediate_logits[-1], info


model = SpectralGaugeCLM(config).to(device)
n_params = sum(p.numel() for p in model.parameters())
n_attn_params = sum(
    sum(p.numel() for p in blk.seq_langevin.score_fn.subbundle_attn.parameters())
    for blk in model.blocks
)
print(f"Total parameters: {n_params:,}")
print(f"Attention parameters: {n_attn_params:,} ({100*n_attn_params/n_params:.1f}% of total)")
print(f"\n4 blocks x {config.langevin_steps} Langevin steps x {config.n_subbundles} subbundles")
print(f"Each subbundle: top-{config.attn_top_k} sparse attention ({config.subbundle_dim}d QKV)")

Total parameters: 3,736,112
Attention parameters: 98,304 (2.6% of total)

4 blocks x 6 Langevin steps x 8 subbundles
Each subbundle: top-8 sparse attention (32d QKV)


---
## 7. Training

In [27]:
@torch.no_grad()
def estimate_loss(model, cfg):
    model.eval()
    results = {}
    for name, sd in [("train", train_data), ("val", val_data)]:
        tot_ce, tot_ok, tot_n, tot_sp = 0., 0, 0, 0.
        bces = [0.] * cfg.n_blocks
        for _ in range(cfg.eval_steps):
            b = get_batch(sd, cfg)
            logits, info = model(b)
            tgt = b[:, 1:]
            ce = F.cross_entropy(logits.reshape(-1, cfg.vocab_size), tgt.reshape(-1))
            tot_ce += ce.item()
            tot_ok += (logits.argmax(-1) == tgt).sum().item()
            tot_n += tgt.numel()
            tot_sp += info["mean_sparsity"]
            for i, bl in enumerate(info["intermediate_logits"]):
                bces[i] += F.cross_entropy(bl.reshape(-1, cfg.vocab_size), tgt.reshape(-1)).item()
        n = cfg.eval_steps
        results[name] = {"ce": tot_ce/n, "acc": tot_ok/tot_n, "sparsity": tot_sp/n,
                         "block_ces": [c/n for c in bces]}
    model.train()
    return results


# v9: No aggressive LR decay. Model is not overfitting — don't choke it.
# Constant LR until overfitting appears, then we can add decay.
optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate, weight_decay=0.01)

history = {
    "step": [], "train_ce": [], "val_ce": [], "train_acc": [], "val_acc": [],
    "train_sparsity": [], "val_sparsity": [],
    "train_bpc": [], "val_bpc": [],
    "block_val_ces": [[] for _ in range(config.n_blocks)], "lr": [],
}

print(f"Training v9 (geometric attention) on Tiny Shakespeare")
print(f"Steps: {config.max_steps}, Batch: {config.batch_size}, Seq: {config.seq_len}")
print("=" * 70)

model.train()
pbar = tqdm(range(config.max_steps), desc="Training", unit="step")

for step in pbar:
    if step % config.eval_interval == 0:
        res = estimate_loss(model, config)
        tr, vl = res["train"], res["val"]
        history["step"].append(step)
        for k in ["ce", "acc", "sparsity"]:
            history[f"train_{k}"].append(tr[k])
            history[f"val_{k}"].append(vl[k])
        history["train_bpc"].append(tr["ce"] / math.log(2))
        history["val_bpc"].append(vl["ce"] / math.log(2))
        history["lr"].append(optimizer.param_groups[0]["lr"])
        for i in range(config.n_blocks):
            history["block_val_ces"][i].append(vl["block_ces"][i])
        tqdm.write(
            f"Step {step:5d} | Train CE: {tr['ce']:.3f} | Val CE: {vl['ce']:.3f} | "
            f"Val BPC: {vl['ce']/math.log(2):.2f} | Val PPL: {math.exp(vl['ce']):.1f} | "
            f"Val Acc: {vl['acc']:.1%} | Sp: {vl['sparsity']:.1%}"
        )

    batch = get_batch(train_data, config)
    optimizer.zero_grad()
    logits, info = model(batch)
    targets = batch[:, 1:]

    ce_loss = sum(
        w * F.cross_entropy(bl.reshape(-1, config.vocab_size), targets.reshape(-1))
        for bl, w in zip(info["intermediate_logits"], info["block_loss_weights"])
    ) / info["block_loss_weights"].sum()

    dcl = 0.
    nd = 0
    for blk in model.blocks:
        for d in blk.memory_bank.dictionaries:
            Dn = F.normalize(d, dim=-1); g = Dn @ Dn.T
            dcl += (g - torch.eye(g.size(0), device=g.device)).pow(2).mean()
            nd += 1
    dcl /= max(nd, 1)

    loss = ce_loss + 0.1 * dcl
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

# Final eval
res = estimate_loss(model, config)
tr, vl = res["train"], res["val"]
history["step"].append(config.max_steps)
for k in ["ce", "acc", "sparsity"]:
    history[f"train_{k}"].append(tr[k])
    history[f"val_{k}"].append(vl[k])
history["train_bpc"].append(tr["ce"] / math.log(2))
history["val_bpc"].append(vl["ce"] / math.log(2))
history["lr"].append(optimizer.param_groups[0]["lr"])
for i in range(config.n_blocks):
    history["block_val_ces"][i].append(vl["block_ces"][i])

print("=" * 70)
print(f"Final | Val CE: {vl['ce']:.3f} | Val BPC: {vl['ce']/math.log(2):.2f} | "
      f"Val Acc: {vl['acc']:.1%} | Val PPL: {math.exp(vl['ce']):.1f}")

Training v9 (geometric attention) on Tiny Shakespeare
Steps: 10000, Batch: 16, Seq: 64


Training:   0%|          | 0/10000 [00:00<?, ?step/s]

Step     0 | Train CE: 5.600 | Val CE: 5.596 | Val BPC: 8.07 | Val PPL: 269.4 | Val Acc: 0.1% | Sp: 0.1%
Step   200 | Train CE: 2.404 | Val CE: 2.387 | Val BPC: 3.44 | Val PPL: 10.9 | Val Acc: 31.0% | Sp: 0.1%
Step   400 | Train CE: 2.203 | Val CE: 2.199 | Val BPC: 3.17 | Val PPL: 9.0 | Val Acc: 35.3% | Sp: 0.1%
Step   600 | Train CE: 2.102 | Val CE: 2.146 | Val BPC: 3.10 | Val PPL: 8.6 | Val Acc: 36.9% | Sp: 0.2%
Step   800 | Train CE: 2.019 | Val CE: 2.090 | Val BPC: 3.02 | Val PPL: 8.1 | Val Acc: 38.0% | Sp: 0.2%


KeyboardInterrupt: 

---
## 8. Diagnostics

In [ ]:
steps = history["step"]
fig, axes = plt.subplots(3, 3, figsize=(18, 15))

axes[0,0].plot(steps, history["train_ce"], 'b-', label='Train')
axes[0,0].plot(steps, history["val_ce"], 'r-', label='Val')
axes[0,0].set_title('Cross-Entropy'); axes[0,0].legend(); axes[0,0].grid(True, alpha=0.3)

axes[0,1].plot(steps, history["train_bpc"], 'b-', label='Train')
axes[0,1].plot(steps, history["val_bpc"], 'r-', label='Val')
axes[0,1].set_title('Bits per Character'); axes[0,1].legend(); axes[0,1].grid(True, alpha=0.3)

vp = [math.exp(c) for c in history["val_ce"]]
tp = [math.exp(c) for c in history["train_ce"]]
axes[0,2].plot(steps, tp, 'b-', label='Train'); axes[0,2].plot(steps, vp, 'r-', label='Val')
axes[0,2].set_title('Perplexity'); axes[0,2].legend(); axes[0,2].grid(True, alpha=0.3)

axes[1,0].plot(steps, history["train_acc"], 'b-', label='Train')
axes[1,0].plot(steps, history["val_acc"], 'r-', label='Val')
axes[1,0].set_title('Accuracy'); axes[1,0].legend(); axes[1,0].grid(True, alpha=0.3)

gap = [v-t for v,t in zip(history["val_ce"], history["train_ce"])]
axes[1,1].plot(steps, gap, 'purple', lw=2)
axes[1,1].set_title('Generalization Gap'); axes[1,1].grid(True, alpha=0.3)

axes[1,2].plot(steps, history["val_sparsity"], 'r-')
axes[1,2].set_title('Sparsity'); axes[1,2].grid(True, alpha=0.3)

colors = plt.cm.viridis(np.linspace(0.2, 0.9, config.n_blocks))
for i in range(config.n_blocks):
    axes[2,0].plot(steps, history["block_val_ces"][i], color=colors[i], label=f'Block {i}')
axes[2,0].set_title('Per-Block Val CE'); axes[2,0].legend(); axes[2,0].grid(True, alpha=0.3)

axes[2,1].plot(steps, history["lr"], 'g-', lw=2)
axes[2,1].set_title('Learning Rate'); axes[2,1].grid(True, alpha=0.3)

# Learned attention schedule vs context schedule (block 0)
with torch.no_grad():
    sf = model.blocks[0].seq_langevin.score_fn
    ctx_a = torch.sigmoid(sf.step_context_logits).cpu().numpy()
    attn_a = torch.sigmoid(sf.step_attn_logits).cpu().numpy()
axes[2,2].plot(ctx_a, 'b-o', ms=4, label='Causal conv')
axes[2,2].plot(attn_a, 'r-o', ms=4, label='Attention')
axes[2,2].set_xlabel('Langevin Step'); axes[2,2].set_ylabel('Force Strength')
axes[2,2].set_title('Learned Force Schedule (Block 0)\n(per Langevin step)')
axes[2,2].legend(); axes[2,2].grid(True, alpha=0.3); axes[2,2].set_ylim(0, 1)

plt.suptitle('v9: Geometric Attention — Full Diagnostics', fontsize=15, fontweight='bold')
plt.tight_layout(); plt.show()

print(f"\nFinal Val BPC: {history['val_bpc'][-1]:.2f} | Val Acc: {history['val_acc'][-1]:.1%} | "
      f"Val PPL: {vp[-1]:.1f}")
print(f"v8.2 baseline: BPC 2.65 | Acc 45% | PPL 6.3")

---
## 9. Text Generation

In [ ]:
@torch.no_grad()
def generate_text(model, prompt_str, max_new=200, temperature=0.8):
    model.eval()
    cfg = model.cfg
    ids = torch.tensor(encode(prompt_str), dtype=torch.long).unsqueeze(0).to(device)
    for _ in range(max_new):
        ctx = ids[:, -cfg.max_seq_len:] if ids.size(1) >= cfg.max_seq_len else ids
        logits, _ = model(ctx)
        probs = F.softmax(logits[:, -1, :] / temperature, dim=-1)
        ids = torch.cat([ids, torch.multinomial(probs, 1)], dim=1)
    return decode(ids[0].tolist())


print("=" * 60)
print("TEXT GENERATION (v9, temperature=0.8)")
print("=" * 60)
for p in ["ROMEO:\n", "To be or not to ", "The king ", "O, "]:
    print(f"\n--- Prompt: {repr(p)} ---")
    print(generate_text(model, p))

---
## 10. Summary

### v9: Geometric Attention

Added per-subbundle sparse attention as the 4th force in the Langevin  
score function, operating inside the settling loop.

```
Score = Hopfield_grad (settle)
      + causal_conv_force (smooth local context)
      + inhibition (competition)
      + ATTENTION_force (selective cross-position retrieval)  ← NEW
```

Each subbundle independently computes sparse top-k causal attention  
over past positions, retrieving content from the most geometrically  
aligned tokens. This gives the model content-dependent cross-position  
selectivity — the capability that every previous version lacked.

The causal conv and attention are complementary:  
- Conv: smooth baseline, position-weighted, content-independent  
- Attention: sharp retrieval, content-addressed, selective

Both are scheduled per Langevin step (high early, low late)  
and both operate inside the settling loop, refining as states sharpen.

---
*v9 notebook by David Ledbetter, 03/14/2026.*  
*Geometric attention: per-subbundle manifold twist detection.*